In [1]:
import os
import sys

current_path = os.path.abspath(os.getcwd())
base_path = os.path.abspath(os.path.join(current_path, '../../../'))
dfield_approach_elastix_path = os.path.abspath(os.path.join(current_path, '../'))
sys.path.append(base_path)
sys.path.append(dfield_approach_elastix_path)

import numpy as np
import matplotlib.pyplot as plt  
import pandas as pd
from tqdm import tqdm
import pandas as pd
from IPython.display import display


import project_images_loader as pil
import utils.util as util
import utils.functions as fc
import utils.nifti_functions as nfc

import constants as cts
import dfield_applier as dfield_applier
import utils.util_freesurfer_segmentation as util_freesurfer

import patches_controller as patches_controller
np.set_printoptions(precision=4)



In [2]:
def visualize_patch(patch_size=32, simple=True):
    %matplotlib widget
    mask_target_path_name = pil.get_segmentation_path_name(3, "00039", simple=simple, absolute=True)
    mask_target, _ = nfc.load_nifti(mask_target_path_name)
    mask_target = util_freesurfer.seg_to_rgbseg(mask_target)

    pinit = 100
    pend = pinit + patch_size
    mask_target[pinit:pend, pinit:pend, pinit:pend] = 0
    fc.imgshow_3D("", mask_target)

# visualize_patch(16, simple=False)
# visualize_patch(16, simple=True)

In [3]:

def compute_tissue_weigthed_dice(patch_target, patch_data):
    tissue_in_target, nb_tissue_voxels_in_target = np.unique(patch_target, return_counts=True)
    tissue_in_data, nb_tissue_voxels_in_data = np.unique(patch_target, return_counts=True)

    tissue_volume_in_target = nb_tissue_voxels_in_target / np.prod(patch_target.shape)
    tissue_volume_in_data = nb_tissue_voxels_in_data / np.prod(patch_data.shape)

    weighted_sum = 0
    total_tissue_weight = 0
    for i, tissue_index in enumerate(tissue_in_target):
        if tissue_index in tissue_in_data:
            tissue_weight_target = tissue_volume_in_target[i]

            i_data = np.where(tissue_in_data == tissue_index)[0][0]
            tissue_weight_data = tissue_volume_in_data[i_data]

            mask1 = patch_target == tissue_index
            mask2 = patch_data == tissue_index

            tissue_dice = util.compute_dice_coefficient(mask1, mask2)
            # weighted_sum += (tissue_weight_target + tissue_weight_data) * tissue_dice
            # tissue weigtehd version
            tissue_weight = 0
            # if tissue_index == 4 or tissue_index == 43:
            #     tissue_weight = 10
            # elif tissue_index == 2 or tissue_index == 41:
            #     tissue_weight = 0.01
            weighted_sum += (tissue_weight + tissue_weight_target + tissue_weight_data) * tissue_dice
            total_tissue_weight += tissue_weight

    dice = weighted_sum / (2+total_tissue_weight)
    # print("Dice: ", dice)
    return dice



In [4]:
%matplotlib widget

def sort_array_by_columns(array):
    ## Slow way to sort the array by columns
    # array_sorted = np.zeros(array.shape)
    # for patch_index in range(len(array)):
    #     sorted_indices = np.argsort(array[:, patch_index, 1])[::-1]
    #     array_sorted[:, patch_index, :] = array[sorted_indices, patch_index, :]

    ## Fats way to sort the array by columns
    sorted_indices = np.argsort(array[:, :, 1], axis=0)[::-1]
    array_sorted = array[sorted_indices, np.arange(array.shape[1]), :]

    return array_sorted

def compute_patches_similarity_table(img_target, ids_dataset, patch_shape=(32,32,32), extraction_step=(16,16,16), similarity_metric="dice", simple_mask=False):
    nb_max_data_images = len(ids_dataset)
    
    # 1. extract patches
    patches_target = patches_controller.extract_patches(img_target, patch_shape, extraction_step)
 
    # 2. compute dice for each patch and each image
    similarity_patches_table = np.zeros((nb_max_data_images, len(patches_target), 2))
    used_ids = []

    img_index = 0
    print("comuting dice for each patch and each image")
    for id_data in tqdm(ids_dataset):
        mask_data, _ = nfc.load_nifti(pil.get_segmentation_path_name(3, id_data, simple=simple_mask, absolute=True))

        if similarity_metric == "dice":
            patches_data = patches_controller.extract_patches(mask_data, patch_shape, extraction_step)
        elif similarity_metric == "ssim":
            img_data, _ = nfc.load_nifti(pil.get_registered_to_template_image_path_name(3, id_data, registration_to=pil.REGISTRATION_TO_MNI, absolute=True))
            img_data = util.skull_stripping_and_normalize(img_data, mask_data)
            patches_data = patches_controller.extract_patches(img_data, patch_shape, extraction_step)
            
            # fc.imgshow_3D("", img_data)
            # return None, None
        
        for patch_index, patch_target in enumerate(patches_target):
            if np.any(patch_target > 0) and np.any(patches_data[patch_index] > 0):
                similarity_patches_table[img_index, patch_index, 0] = img_index
                if similarity_metric == "dice":
                    similarity_patches_table[img_index, patch_index, 1] = compute_tissue_weigthed_dice(patch_target, patches_data[patch_index])
                elif similarity_metric == "ssim":
                    similarity_patches_table[img_index, patch_index, 1] = util.compute_ssim(patch_target, patches_data[patch_index])
            else:
                similarity_patches_table[img_index, patch_index, 0] = -1
            
        img_index += 1
        used_ids.append(id_data)

        if img_index == nb_max_data_images:
            break

    return similarity_patches_table, used_ids



def compute_patches_similarity_table_combined_metrics(img_target, seg_target, ids_dataset, patch_shape=(32,32,32), extraction_step=(16,16,16), simple_mask=False):
    nb_max_data_images = len(ids_dataset)
    
    # 1. extract patches
    patches_img_target = patches_controller.extract_patches(img_target, patch_shape, extraction_step)
    patches_seg_target = patches_controller.extract_patches(seg_target, patch_shape, extraction_step)
 
    # 2. compute dice for each patch and each image
    similarity_patches_table = np.zeros((nb_max_data_images, len(patches_img_target), 2))
    used_ids = []

    img_index = 0
    print("comuting dice for each patch and each image")
    for id_data in tqdm(ids_dataset):
        mask_data, _ = nfc.load_nifti(pil.get_segmentation_path_name(3, id_data, simple=simple_mask, absolute=True))
        img_data, _ = nfc.load_nifti(pil.get_registered_to_template_image_path_name(3, id_data, registration_to=pil.REGISTRATION_TO_MNI, absolute=True))
        img_data = util.skull_stripping_and_normalize(img_data, mask_data)

        patches_img_data = patches_controller.extract_patches(img_data, patch_shape, extraction_step)
        patches_seg_data = patches_controller.extract_patches(mask_data, patch_shape, extraction_step)
        
        for patch_index, (patch_img_target, patch_seg_target) in enumerate(zip(patches_img_target, patches_seg_target)):
            if np.any(patch_seg_target > 0) and np.any(patches_seg_data[patch_index] > 0):
                similarity_patches_table[img_index, patch_index, 0] = img_index

                dice_metric = compute_tissue_weigthed_dice(patch_seg_target, patches_seg_data[patch_index])
                ssim_metric = util.compute_ssim(patch_img_target, patches_img_data[patch_index])
                similarity_patches_table[img_index, patch_index, 1] = (dice_metric + ssim_metric) / 2

            else:
                similarity_patches_table[img_index, patch_index, 0] = -1
            
        img_index += 1
        used_ids.append(id_data)

        if img_index == nb_max_data_images:
            break

    return similarity_patches_table, used_ids

In [5]:
def compute_pathc_based_dfield(similarity_patches_table, target_ids, nb_best_images=10, patch_shape=(32,32,32), extraction_step=(16,16,16)):
    # 1. sort patches_dice by columns individually to obtain the best image for each patch
    similarity_patches_table = sort_array_by_columns(similarity_patches_table)
    most_similar_patches = similarity_patches_table[:nb_best_images]

    # 2. compute best patches weights (l1 normalized dice scores)
    l1_norm = np.linalg.norm(most_similar_patches[:, :, 1], ord=1, axis=0)
    l1_norm[l1_norm == 0] = 1
    weights_columns = most_similar_patches[:,:,1] / l1_norm
    most_similar_patches = np.concatenate((most_similar_patches, weights_columns[:, :, np.newaxis]), axis=2)

    # 3. ontain dfields index to load
    dfields_data_to_load = np.unique(most_similar_patches[:, :, 0])
    dfields_data_to_load = [int(i) for i in dfields_data_to_load if i >= 0]
    # print("dfields_data_to_load", [target_ids[i] for i in dfields_data_to_load])

    # # print("most_similar_patches")
    # sum_most_similar_patches = np.sum(most_similar_patches[:,:,1], axis=0)
    # print("Min sum of dice scores for the best patches")
    # for i in range(nb_best_images):
    #     print(most_similar_patches[i, np.argmin(sum_most_similar_patches), :])
    
    # print("Max sum of dice scores for the best patches")
    # for i in range(nb_best_images):
    #     print(most_similar_patches[i, np.argmax(sum_most_similar_patches), :])
        
    # 4. compute dfield patch weighted mean
    pathces_shape = (similarity_patches_table.shape[1], *patch_shape)
    patches_dfield_target_x = np.zeros(pathces_shape)
    patches_dfield_target_y = np.zeros(pathces_shape)
    patches_dfield_target_z = np.zeros(pathces_shape)
    
    affine = None
    dfiled_shape = None
    # 5. recontruct the dfield from the patches
    # looping over the best dfields in order to be faster
    print("constructing target dfield from the best dfields")
    for img_data_index in tqdm(dfields_data_to_load):
        dfield_data_path_name = cts.get_image_path_name(target_ids[img_data_index], cts.PFILE_0038, cts.INAME_DFIELD, subfolder=cts.FOLDER_REGISTRATIONS)
        dfield_data, affine = nfc.load_nifti(dfield_data_path_name)
        dfield_data = dfield_data[:,:,:,0]
        
        dfield_data_x, dfield_data_y, dfield_data_z = dfield_data[..., 0], dfield_data[..., 1], dfield_data[..., 2]
        patches_dfield_data_x = patches_controller.extract_patches(dfield_data_x, patch_shape, extraction_step)
        patches_dfield_data_y = patches_controller.extract_patches(dfield_data_y, patch_shape, extraction_step)
        patches_dfield_data_z = patches_controller.extract_patches(dfield_data_z, patch_shape, extraction_step)

        # looping over the patches
        for patch_index in range(most_similar_patches.shape[1]):
            # looping over the dfields needed to compute the weighted mean for the current patch
            current_patch_table = most_similar_patches[:, patch_index, :]
            for img_index, img_dice, img_weight  in current_patch_table:
                # check the loaded best dfield is part of the weigthted mean of the current patch
                if img_index == img_data_index:
                    # add the dfield to the weighted mean of the current patch
                    patches_dfield_target_x[patch_index] += img_weight * patches_dfield_data_x[patch_index]
                    patches_dfield_target_y[patch_index] += img_weight * patches_dfield_data_y[patch_index]
                    patches_dfield_target_z[patch_index] += img_weight * patches_dfield_data_z[patch_index]

    # 7. Reoconstruct the dfield from the patches
    dfiled_shape = dfield_data.shape[:3]
                    
    dfield_target_x = patches_controller.perform_voting(patches_dfield_target_x, patch_shape, dfiled_shape, extraction_step, extraction_step)
    dfield_target_y = patches_controller.perform_voting(patches_dfield_target_y, patch_shape, dfiled_shape, extraction_step, extraction_step)
    dfield_target_z = patches_controller.perform_voting(patches_dfield_target_z, patch_shape, dfiled_shape, extraction_step, extraction_step)

  
    dfield_target = np.stack((dfield_target_x, dfield_target_y, dfield_target_z), axis=-1)
    dfield_target = np.expand_dims(dfield_target, axis=-2)
    return dfield_target, affine


### Create patch based dfields

In [6]:

def create_patch_base_dfield(id_target, matchin_ids, nb_images_list, patch_shape=(32,32,32), extraction_step=(16,16,16), similarity_metric='dice'):
    ids_dataset = matchin_ids[:300]
    simple_segmentation = False
    
    mask_target, _ = nfc.load_nifti(pil.get_segmentation_path_name(3, id_target, simple=simple_segmentation, absolute=True))
    if similarity_metric == "dice":
        img_target = mask_target
    elif similarity_metric == "ssim" or similarity_metric == "dice_ssim":
        img_target, _ = nfc.load_nifti(pil.get_registered_to_template_image_path_name(3, id_target, pil.REGISTRATION_TO_MNI, absolute=True))
        img_target = util.skull_stripping_and_normalize(img_target, mask_target)

    if similarity_metric == "dice_ssim":
        similarity_patches_table, used_ids = compute_patches_similarity_table_combined_metrics(img_target, mask_target, ids_dataset, patch_shape=patch_shape, extraction_step=extraction_step, simple_mask=simple_segmentation)
    else:
        similarity_patches_table, used_ids = compute_patches_similarity_table(img_target, ids_dataset, patch_shape=patch_shape, extraction_step=extraction_step, similarity_metric=similarity_metric, simple_mask=simple_segmentation)

    for nb_images in nb_images_list:
        # dfield_target, affine = compute_pathc_based_dfield(similarity_patches_table, used_ids, nb_best_images=nb_images, patch_shape=patch_shape, extraction_step=extraction_step)
        dfield_target, affine = compute_pathc_based_dfield(similarity_patches_table, used_ids, nb_best_images=nb_images, patch_shape=patch_shape, extraction_step=extraction_step)
        nfc.save_nifti(dfield_target, affine, cts.get_patch_based_dfield_path_name(id_target, similarity_metric.upper(), patch_shape[0], cts.PFILE_0038, nb_images))



matchin_ids = pil.get_matching_ids()
id_indexs = [301,302,303,304,305,306,307]
# id_indexs = [302]


# patch_shape = (64,64,64)
# extraction_step = (32,32,32)
# patch_shape = (32,32,32)
# extraction_step = (16,16,16)
patch_shape = (16,16,16)
extraction_step = (8,8,8)
# patch_shape = (8,8,8)
# extraction_step = (4,4,4)

similarity_metric='dice'
similarity_metric2='ssim'

# for id_index in id_indexs:
#     id_target = matchin_ids[id_index]
#     create_patch_base_dfield(id_target, matchin_ids, [10,100], patch_shape=patch_shape, extraction_step=extraction_step, similarity_metric=similarity_metric)
#     create_patch_base_dfield(id_target, matchin_ids, [10,100], patch_shape=patch_shape, extraction_step=extraction_step, similarity_metric=similarity_metric2)
